# Trade Area Modeling
According to [Market Business News](https://marketbusinessnews.com/financial-glossary/trade-area-definition-meaning/), a trade (or market) area is the geographical area  where all or most of a business' sales volume occurs. It can be used to make decisions about the optimum location for business growth and expansion.

There are different trade area model and in this use case, we will be using the widely accepted [Huff model](https://en.wikipedia.org/wiki/Huff_model). The Huff model uses the **distance** between customers and existing business locations, or locations that are planned for the future, along with the **attractiveness of those locations**, to determine the likelihood of customers visiting those locations.

## Case Study
Lotionfy is a fictional beauty and personal care brand that has thrived as an exclusively online business for the past two years. They serve a predominantly US market, have done some market research and this is what they have discovered.
1. While the e-commerce share of total retail sales continues to rise, in-store shopping is still the preferred method for most US cusstomers and makes up a very large chunk of total retail sales. [[Reference](https://capitaloneshopping.com/research/online-vs-in-store-shopping-statistics/)]
2. More online shoppers are opting to pick up orders in-store and shoppers are generally returning to stores; retailers are reaching customers with small-format stores to encourage impulse purchases and reach new markets, among other benefits[[Reference](https://www2.deloitte.com/us/en/pages/consulting/articles/q1-2023-consumer-trends-report.html)]
3. Online businesses are developing physical footprints to keep their customers loyal and give them additional touchpoints for the brand; customers themselves now prefer omnichannel experiences [[Reference](https://www.inc.com/rebecca-deczynski/the-future-of-retail-isnt-direct-to-consumer-brands-embracing-brick-mortar-2023.html)]

In light of these, Lotionfy has decided to build their physical presence. To start, they have also decided to leverage partnerships with already existing [top department stores](https://www.junglescout.com/wp-content/uploads/2023/09/Jungle-Scout-Consumer-Trends-Report-Q3-2023.pdf) in the US like Walmart, Target, Kohl's etc. They will be leveraging the small-format store approach by drilling down to neighborhood locations.
They have decided to use a trade area model to know which cities to focus on and which store locations should be their areas of focus.

### Developing a fictional customer base for the brand
To begin with, the [Statistical Atlas](https://statisticalatlas.com/United-States/Overview) website is scraped to get the list of some cities in the US. These cities are further scraped to get a random list of neighborhoods and their population. It is quite unlikely for the entire population of a neighborhood to be part of the customer base so the customer population in the neighborhoods will be randomly generated, ensuring that it is no more than the total population.

_Please note that this scraping was done in November 2023. After the time of this scraping, the website UI may have been changed by the web developers and the code may need to be changed accordingly._

#### Modules and Clients

In [1]:
# import relevant packages
from timezonefinder import TimezoneFinder
from geopy.geocoders import Nominatim
from shapely.geometry import Polygon
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import osmnx as ox
import googlemaps
import datetime
import requests
import polyline
import random
import pyproj
import pytz
import utm
import os

In [2]:
# load environment variables
load_dotenv()
# create a random seed for reproducibility
random_seed = 42

In [50]:
# create Nominatim client
nominatim_locator = Nominatim(user_agent='trade_area_model', timeout=30)
# initialize googlemaps client - use your own key
gmaps = googlemaps.Client(key=os.getenv('google_api_key'))
# store OpenCage API key - use your own key
opencage_api_key = os.getenv('opencage_api_key')
# create timezone finder
tf = TimezoneFinder()

#### Creating cities

In [4]:
# create a list to store html list elements, and major cities and places
html_list, city_list = [], []
# initiate the http request to the website for scraping
url = f'https://statisticalatlas.com/United-States/Overview'
response = requests.get(url)
# if the http response is successful
if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser') # parse html
    # find all div elements with the table class
    contents = soup.find_all('div', class_='info-table-contents-div')
    # if the hyperlink confirms that it is a city, add it to the html list
    for content in contents:
        hyperlink = content.find('a').get('href')
        if 'place' in hyperlink:
            cities = content.text.strip()
            html_list.append(cities)
            # add the cities in each html list element to the city list
            for cities in html_list:
                cities = cities.split(', ')
                city_list += cities
else: # if the http request failed
    print(f'Request failed: {response.status_code}')

In [5]:
# create a dictionary for cities and states
city_state_dict = {}
# for each city
for city in city_list:
    # geocode its location
    location = nominatim_locator.geocode(city).raw
    latitude, longitude = location['lat'], location['lon']
    state = location['display_name'].split(', ')[-2]
    # add the city, state, lat and long to the dictionary
    city_state_dict[city] = [state, latitude, longitude]

In [6]:
# create a dataframe of the customer cities
customer_cities = pd.DataFrame(city_state_dict.items(), columns=['city', 'state'])
customer_cities[['state','latitude','longitude']] = customer_cities['state'].apply(pd.Series)

In [7]:
# save customer cities data
customer_cities.to_csv('data/customers/customer_cities.csv', index=False)

#### Creating city neighborhoods

In [8]:
# create a city location column
customer_cities['location'] = customer_cities[['city','state']].agg(lambda x: ', '.join(x), axis=1)

In [9]:
# create a dictionary to store the list of neighborhoods in each city
city_neighborhoods = {}

# iterating through each row
for row in customer_cities.index:
    # store the state name, city and location in variables
    # rename certain cities based on the names used in the website we are web scraping from
    state = customer_cities.loc[row,'state'].replace(' ','-')
    city = customer_cities.loc[row,'city'].replace(' ','-')
    if city == 'Boise' and state == 'Idaho': city = 'Boise-City'
    elif city == 'Indianapolis' and state == 'Indiana': city = 'Indianapolis-city-(balance)'
    elif city == 'Louisville' and state == 'Kentucky': city = 'Louisville/Jefferson-County-metro-government-(balance)'
    elif city == 'Nashville-Davidson' and state == 'Tennessee': city = 'Nashville-Davidson-metropolitan-government-(balance)'
    location = customer_cities.loc[row,'location']
    # create the url to scrape from and initiate a http request
    url = f'https://statisticalatlas.com/place/{state}/{city}/Overview'
    response = requests.get(url)
    # if the http response is successful
    if response.status_code == 200:
        try: # try this
            soup = BeautifulSoup(response.text, 'html.parser') # parse html
            # find the div and class elements that should contain the list of neighborhoods
            # confirm from the hyperlinks that it contains neighborhood data
            contents = soup.find_all('div', class_='info-table-contents-div')
            for content in contents:
                hyperlink = content.find('a').get('href')
                if 'neighborhood' in hyperlink:
                    neighborhoods = content.text.strip()
                    # store the neighborhood data as values in the dictionary with the location as keys
                    city_neighborhoods[location] = neighborhoods
        except Exception as error: # if there is an error, print the error and continue execution
            print(url); print(str(error)); pass
    else: # if the response failed, print the status code
        print("Failed to get data, status code:",response.status_code)
    response.close()

In [10]:
# create a dataframe with the locations (city and state) and neighborhoods
city_communities = pd.DataFrame(list(city_neighborhoods.items()), columns=['location', 'neighborhoods'])

In [11]:
# split the neighborhoods by the delimiter
random.seed(random_seed)
city_communities['neighborhoods'] = city_communities['neighborhoods'].str.split(', ')#\
    #.apply(lambda x: random.sample(x, min(30, len(x))))
# explode the list of neighborhoods so each has its unique row
city_communities = city_communities.explode('neighborhoods', ignore_index=True)

In [12]:
# get the geopolygon of the neighborhood boundaries with this user-defined function
def get_neighborhood_boundaries(neighborhood):
    boundary = None
    try:
        city = ox.geocode_to_gdf(neighborhood)
        city = city.loc[(city['geometry'].astype(str).str.contains('POLYGON')) 
                        | (city['geometry'].astype(str).str.contains('MULTIPOLYGON'))]
        boundary = city.loc[0,'geometry']
        print(neighborhood, end='; ')
    except: pass
    return boundary

In [13]:
# create a columm for neighborhood boundaries
city_communities['geometry'] = city_communities[['neighborhoods','location']].apply(lambda x:', '.join(x), axis=1)\
    .apply(get_neighborhood_boundaries)

Academy Hills Park, Albuquerque, New Mexico; Academy Park, Albuquerque, New Mexico; Alamosa, Albuquerque, New Mexico; Altura, Albuquerque, New Mexico; Alvarado Park, Albuquerque, New Mexico; Avalon, Albuquerque, New Mexico; Barelas, Albuquerque, New Mexico; Bear Canyon, Albuquerque, New Mexico; Bel-Air, Albuquerque, New Mexico; Boyds-Leslie Park, Albuquerque, New Mexico; Campus, Albuquerque, New Mexico; Cherry Hills, Albuquerque, New Mexico; Conchas Park, Albuquerque, New Mexico; Crestview Heights, Albuquerque, New Mexico; Del Norte, Albuquerque, New Mexico; Del Rey R, Albuquerque, New Mexico; Downtown, Albuquerque, New Mexico; Eagle Ranch, Albuquerque, New Mexico; Four Hills Village, Albuquerque, New Mexico; Heritage Hills, Albuquerque, New Mexico; Hodgin, Albuquerque, New Mexico; Holiday Park, Albuquerque, New Mexico; Inez, Albuquerque, New Mexico; Jade Park, Albuquerque, New Mexico; Jerry Cline Park, Albuquerque, New Mexico; John Robert, Albuquerque, New Mexico; Las Terrazas, Albuqu

In [14]:
# remove neighborhoods without boundary polygons
city_communities = city_communities[city_communities['geometry'].notna()]

In [15]:
# remove duplicates and cities without listed neighborhoods, if any
city_communities = city_communities[~(city_communities.duplicated()) & (city_communities['neighborhoods'] != '')]

#### Obtaining neighborhood population

In [16]:
# explicitly cast the population column as a string
city_communities.loc[:,'population'] = ''

# iterating through each neighborhood
for row in city_communities.index:
    # store state name, city, and neighborhood in variables
    # replace certain cities based on the names in the website we are web scraping from
    location = city_communities.loc[row,'location'].replace(' ','-').split(',-')
    city, state = location[0], location[1]
    if city == 'Boise' and state == 'Idaho': city = 'Boise-City'
    elif city == 'Indianapolis' and state == 'Indiana': city = 'Indianapolis-city-(balance)'
    elif city == 'Louisville' and state == 'Kentucky': city = 'Louisville/Jefferson-County-metro-government-(balance)'
    elif city == 'Nashville-Davidson' and state == 'Tennessee': city = 'Nashville-Davidson-metropolitan-government-(balance)'
    neighborhood = city_communities.loc[row,'neighborhoods'].replace(' ','-').replace("'",'').replace('.','')
    # create the site url and initiate the http request
    url = f'https://statisticalatlas.com/neighborhood/{state}/{city}/{neighborhood}/Overview'
    response = requests.get(url)
    # if the request is successful
    if response.status_code == 200 and response.history == []:
        try: # try this
            soup = BeautifulSoup(response.text, 'html.parser') # parse html
            # find tr element containing the population number
            population_html = soup.find_all('tr')[-2]
            population = population_html.text.strip()
            # store population in its cell
            city_communities.loc[row, 'population'] = population
        except Exception as error: # for exceptions, print the url, error, and continue execution
            print(url, str(error), end=','); pass
    else: # if the http response fails, print its url, status code, and history
        print("Failed to get data",url,response.status_code, response.history)
    response.close()

Failed to get data https://statisticalatlas.com/neighborhood/Georgia/Atlanta/Hartsfield-Jackson-Atlanta-International-Airport/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Georgia/Atlanta/Parkwood/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Texas/Austin/Barton-Creek/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Maryland/Baltimore/Dundalk-Marine-Terminal/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Maryland/Baltimore/Hawkins-Point/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Maryland/Baltimore/Johns-Hopkins-Homewood/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Massachusetts/Boston/Assembly-Square/Overview 200 [<Response [303]>]
Failed to get data https://statisticalatlas.com/neighborhood/Massachusetts/Bos

In [17]:
# filter out rows with empty population
city_communities = city_communities[city_communities['population'] != '']

In [18]:
# include a date key for versioning purposes
city_communities.loc[:, 'table_date'] = datetime.date.today()

In [19]:
# save this data in csv
city_communities.to_csv('data/all_city_communities.csv', index=False)

In [20]:
# set each city to have no more than 30 neighborhoods/communities
# this step can be skipped
city_communities = city_communities.groupby('location', group_keys=False).apply(lambda x: x.sample(n=min(30, len(x)),
                                                                                       random_state=random_seed))

In [21]:
# save this data in csv
city_communities.to_csv('data/city_communities.csv', index=False)

#### Generating customer population

In [22]:
# create a new column for customer population with reproducible random values less than the total population
np.random.seed(random_seed)
customer_communities = city_communities.copy()
customer_communities['population'] = (customer_communities['population'].str.replace(',','').astype(float)\
                                      * np.random.rand(len(customer_communities))).astype(int)

In [23]:
# save this data in csv
customer_communities.to_csv('data/customers/customer_communities.csv', index=False)

#### Find prime community with most customers

In [65]:
# find city location with the most customers
number_of_customers = customer_communities.groupby('location')['population'].sum()\
    .sort_values(ascending=False)
number_of_customers

location
Los Angeles, California             647180
New York, New York                  374890
Houston, Texas                      369645
Dallas, Texas                       362494
San Jose, California                327914
Boston, Massachusetts               304721
Chicago, Illinois                   213756
Colorado Springs, Colorado          193282
Indianapolis, Indiana               185729
Fresno, California                  166023
Las Vegas, Nevada                   138349
Denver, Colorado                    110162
Portland, Oregon                    108430
Philadelphia, Pennsylvania          106721
Raleigh, North Carolina             104217
San Diego, California                99944
Cleveland, Ohio                      97799
San Francisco, California            91790
Sacramento, California               89501
Washington, District of Columbia     75561
Austin, Texas                        68216
Phoenix, Arizona                     66912
Arlington, Texas                     62421
Co

In [66]:
# select prime city location
prime_location = number_of_customers.index[0]
# select customer neighborhoods in the prime city location
prime_city_neighborhoods = customer_communities.loc[customer_communities['location'] == prime_location,
                                                ['location','neighborhoods']]

### Store Locations
The following department stores in the prime location will be used as potential store locations: Walmart, Target, JCPenny, Kohl's and Marshalls

#### Finding stores

In [51]:
def get_address(geometry):
    lat, lng = geometry.centroid.y, geometry.centroid.x
    result = gmaps.reverse_geocode((lat, lng))
    address = result[0]['formatted_address']
    address = ', '.join(address.split(', ')[:2])
    return address

In [52]:
def get_nominatim_address(gmap_address):
    results = gmaps.places(query=gmap_address)
    geom = results['results'][0]['geometry']['location']
    lat, lng = geom['lat'], geom['lng']
    address = nominatim_locator.reverse((lat, lng)).raw['display_name']
    return address

In [28]:
# check for these department stores on GMaps
# get the OSM features of these stores
department_stores = pd.DataFrame()
dept_store_list = ["Walmart", "Target", "JCPenney", "Kohl's", "Marshalls"]
for name in dept_store_list:
    query = f"{name}, {prime_location}"
    results = gmaps.places(query=query)
    for result in results['results']:
        address = result['formatted_address']
        name = result['name']
        try:
            nominatim_address = get_nominatim_address(address)
            df = ox.features_from_address(nominatim_address, tags = {'shop':['department_store','supermarket']})
            df.loc[:,['gmap_name', 'address']] = name, ', '.join(address.split(', ')[:2])
            department_stores = pd.concat([department_stores, df], axis=0)
        except Exception as error: 
            print(f'Error with {name}; {address}: {error}', end='\n\n'); pass

In [29]:
# list all available department store names
department_store_names = department_stores['name'].dropna().unique().tolist()
# select all stores that contain the target store names
selected_stores = [item for item in department_store_names if any(store in item for store in dept_store_list)]
# keep stores with defined names and geopolygons
department_stores = department_stores.loc[(department_stores['name'].isin(selected_stores)) &
                                          (department_stores['geometry'].astype(str).str.contains('POLYGON')),
                                          ['name','gmap_name','address', 'geometry','building:levels']]
# remove duplicates
department_stores = department_stores.sort_index().reset_index().drop_duplicates('osmid',keep='last')

In [53]:
# get the correct gmaps address of stores near to the gmap store that was also selected by OSM
department_stores.loc[department_stores['name'] != department_stores['gmap_name'], 'address'] = \
    department_stores['geometry'].apply(get_address)
# remove the gmap name from the dataframe
department_stores.drop('gmap_name', axis=1, inplace=True)

In [54]:
# create store locations with name and address
department_stores['store_location'] = department_stores[['name','address']].apply(lambda x:', '.join(x), axis=1)
store_locations = department_stores['store_location'].unique()

In [55]:
department_stores['city_location'] = prime_location

In [56]:
department_stores.to_csv('data/stores/store_data.csv', index=False)

#### Distance matrix and travel time indices

In [57]:
# get timezone of prime location
geocode_prime_location = gmaps.geocode(prime_location)
prime_location_geom = geocode_prime_location[0]['geometry']['location']
timezone = tf.timezone_at(lat=prime_location_geom['lat'], lng=prime_location_geom['lng'])

In [67]:
for store in store_locations:
    # create column for store path, distance and time from neighborhood
    prime_city_neighborhoods.loc[:, 'distance '+store] = ''
    prime_city_neighborhoods.loc[:, 'time '+store] = ''
    prime_city_neighborhoods.loc[:, 'path '+store] = ''
    # iterating through each neighborhood
    for row in prime_city_neighborhoods.index:
        # store the customer location and its geolocation in variables
        neighborhood = prime_city_neighborhoods.loc[row, 'neighborhoods']
        customer_location = neighborhood + ', ' + prime_location
        # set gmaps departure time
        try:
            departure_time = datetime.datetime.now(pytz.timezone(timezone)).replace(hour=17, minute=0, second=0)
            # get the path, distance and travel time and add them to the neighborhood df
            geocode_result = gmaps.directions(customer_location, store, mode='driving', traffic_model='pessimistic',
                                            units='metric', departure_time=departure_time)
        except:
            day = datetime.date.today().day + 1
            departure_time = datetime.datetime.now(pytz.timezone(timezone)).replace(day=day, hour=17, minute=0, second=0)
            # get the path, distance and travel time and add them to the neighborhood df
            geocode_result = gmaps.directions(customer_location, store, mode='driving', traffic_model='pessimistic',
                                            units='metric', departure_time=departure_time)
        dist_duration = geocode_result[0]['legs'][0]
        distance = dist_duration['distance']['text']
        time = dist_duration['duration_in_traffic']['text']
        encoded_path = geocode_result[0]['overview_polyline']['points']
        path = polyline.decode(encoded_path)
        prime_city_neighborhoods.loc[row, 'distance '+store] = distance
        prime_city_neighborhoods.loc[row, 'time '+store] = time
        prime_city_neighborhoods.loc[row, 'path '+store] = str(path)

C:\Users\iolowoye\AppData\Local\Temp\ipykernel_22088\1056731747.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prime_city_neighborhoods.loc[:, 'distance '+store] = ''
C:\Users\iolowoye\AppData\Local\Temp\ipykernel_22088\1056731747.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  prime_city_neighborhoods.loc[:, 'time '+store] = ''
C:\Users\iolowoye\AppData\Local\Temp\ipykernel_22088\1056731747.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which 

In [69]:
# create a function to convert distances  to km
def convert_to_km(distance_str):
    value, unit = float(distance_str.split()[0]), distance_str.split()[1].lower()
    unit_conversion_dict = {'km':1.0, 'm':0.001}
    return value * unit_conversion_dict[unit]

In [70]:
# create a function to convert time to minutes
def convert_to_mins(time_str):
    total_minutes = 0
    time = time_str.split()
    if 'hours' in time or 'hour' in time: total_minutes += (int(time[0]) * 60) + int(time[2])
    elif 'mins' in time or 'min' in time: total_minutes += int(time[0])
    return total_minutes

##### Distance matrix

In [71]:
# columns for distance matrix
distance_columns = ['neighborhoods'] + [col for col in prime_city_neighborhoods.columns if 'distance' in col]
# create distance matrix
distance_matrix = prime_city_neighborhoods.loc[:, distance_columns]
# remove neighborhood prefixes and suffixes
distance_matrix.columns = distance_matrix.columns.str.removeprefix('distance').str.removesuffix(prime_location).str.strip()\
    .str.removesuffix(',')
# set neighborhoods as index
distance_matrix.set_index('neighborhoods', inplace=True)
# standardize units
for col in distance_matrix.columns:
    try: distance_matrix[col] = distance_matrix[col].map(convert_to_km)
    except Exception as error: print(f'Could not convert for column {col}'); pass

In [72]:
# save data
distance_matrix.to_csv('data/stores/store_distances.csv')

##### Travel time indices

In [73]:
# columns for travel time
time_columns = ['neighborhoods'] + [col for col in prime_city_neighborhoods if 'time' in col]
# create travel time data
travel_time = prime_city_neighborhoods.loc[:, time_columns]
# remove neighborhood prefixes and suffixes
travel_time.columns = travel_time.columns.str.removeprefix('time').str.removesuffix(prime_location).str.strip()\
    .str.removesuffix(',')
# set neighborhoods as index
travel_time.set_index('neighborhoods', inplace=True)
# standardize units
for col in travel_time.columns:
    try: travel_time[col] = travel_time[col].map(convert_to_mins)
    except Exception as error: print(f'Could not convert for column {col}'); pass

In [74]:
# store data
travel_time.to_csv('data/stores/traffic_time_mins.csv')

##### Path

In [75]:
# columns for travel time
path_columns = ['neighborhoods'] + [col for col in prime_city_neighborhoods if 'path' in col]
# create travel time data
path_df = prime_city_neighborhoods.loc[:, path_columns]
# remove neighborhood prefixes and suffixes
path_df.columns = path_df.columns.str.removeprefix('path').str.removesuffix(prime_location).str.strip()\
    .str.removesuffix(',')
# set neighborhoods as index
path_df.set_index('neighborhoods', inplace=True)

In [76]:
# store data
path_df.to_csv('data/stores/path_to_store.csv')

#### Store preferences
These include the travel time indices, median neighborhood income, number of highways, neighborhood design index, size of the store, accessibility (how many drivable and walkable roads are there?), and number of parking space.

In [77]:
# create travel index of store
store_preferences = round(travel_time.mean(numeric_only=True), 1).to_frame(name='avg_travel_time')

In [79]:
# create a list of major highway tags
major_highways = ['motorway', 'trunk', 'primary', 'secondary', 'motorway_link', 'trunk_link',
                  'primary_link', 'secondary_link', 'motorway_junction']

# store the city and state in variables
store_city = prime_location.split(', ')[0].replace(' ','-')
store_state = prime_location.split(', ')[1].replace(' ','-')

# create placeholder columns
store_preferences.loc[:, 'business_communities'] = 0
store_preferences.loc[:, 'highways'] = 0
store_preferences.loc[:, 'design'] = 0
store_preferences.loc[:, 'accessibility'] = 0
store_preferences.loc[:, 'parking_space'] = 1
store_preferences.loc[:, 'store_size'] = 0.0

# create dictionaries for the different dataframes
businesses_dict, highways_dict, design_dict, road_dict, walkways_dict, parking_dict = ({} for _ in range(6))

# iterate through the store locations
for store in store_locations:
    
    store_address = get_nominatim_address(store)
    """ BUSINESS DISTRICTS """
    # find and count number of commercial landspace
    try:
        businesses_df = ox.features_from_address(store_address, tags={'landuse':'commercial'}).reset_index()
        businesses_dict[store] = businesses_df
        businesses = businesses_df['osmid'].nunique()
        store_preferences.loc[store, 'business_communities'] = businesses
    except: pass
    
    """ HIGHWAYS """
    # find and count number of highways
    try:
        highways_df = ox.features_from_address(store_address, tags={'highway':major_highways})
        highways_dict[store] = highways_df
        highways = highways_df['name'].nunique()
        store_preferences.loc[store, 'highways'] = highways
    except: pass
    
    """ MERCHANDIZING DESIGN """
    # find amenities and other neighborhood design touchpoints
    try:
        design_df = ox.features_from_address(store_address, tags={'amenity':True, 'leisure':True,
                                                                  'natural':'beach', 'office':True,
                                                                  'landuse':['commercial','education','fairground',
                                                                             'religious'],'tourism':True})
        design_dict[store] = design_df
        design = design_df.name.nunique()
        store_preferences.loc[store, 'design'] = design
    except: pass
    
    """ ACCESSIBILITY """
    # estimate accessibility
    try:
        road_df = ox.features_from_address(store_address, tags={'highway':['tertiary', 'bus_stop'], 
        'public_transport':True})
        road_dict[store] = road_df
        roads = road_df.name.nunique()
    except: roads = 0; pass
    try:
        walkways_df = ox.features_from_address(store_address, tags={'highway':['footway', 'track','cycleway']}).reset_index()
        walkways_dict[store] = walkways_df
        walkways = walkways_df.osmid.nunique()
    except: walkways = 0; pass
    store_preferences.loc[store, 'accessibility'] = roads + walkways
    
    """ PARKING SPACE """
    # create dataframe to store parking space
    try:
        parking_df = ox.features_from_address(store_address, tags = {'building':'parking', 'amenity':'parking'})
        parking_df = parking_df.loc[parking_df['access'] == 'yes', :].reset_index().drop_duplicates('osmid')
        parking_df['building:levels'] = parking_df['building:levels'].fillna(1).astype(int)
        parking_dict[store] = parking_df
        parking = parking_df['building:levels'].sum(numeric_only=True)
        if parking > 0: store_preferences.loc[store, 'parking_space'] = parking
    except: pass
    
    """ STORE SIZE """
    # select geometry
    store_size_df = department_stores.loc[department_stores['store_location']==store, :]
    geometry = list(store_size_df['geometry'].iloc[0].exterior.coords)
    # reverse the geometry position to be lat-long not long-lat
    geometry_reverse = []
    for coord in geometry:
        latlon = (coord[1], coord[0])
        geometry_reverse.append(latlon)
    # extract utm zone
    zone = utm.from_latlon(geometry_reverse[0][0], geometry_reverse[0][1])
    zone_num = zone[2]
    # get utm coordinates of store geometry
    utm_zone = pyproj.Proj(f"+proj=utm +zone={zone_num} +ellps=WGS84 +units=m")
    utm_coord = [utm_zone(latitude=lat, longitude=long) for lat, long in geometry_reverse]
    # calculate the area of the selected retail store
    area_m2 = Polygon(utm_coord).area
    # count number of building levels
    try: bld_lvls = int(store_size_df['building:levels'].iloc[0])
    except: bld_lvls = 1
    # calculate total store area
    store_size_m2 = round(area_m2 * bld_lvls, 2)
    # add to the store preferences data
    store_preferences.loc[store, 'store_size'] = store_size_m2
    print(f'{store} preferences completed')

Walmart Supercenter, 8500 Washington Blvd, Pico Rivera preferences completed
Walmart, 8333 Van Nuys Blvd, Panorama City preferences completed
Walmart Supercenter, 7250 Carson Blvd, Long Beach preferences completed
Walmart Supercenter, 19821 Rinaldi St, Porter Ranch preferences completed
Target, 8840 Corbin Ave, Northridge preferences completed
Kohl's, 8800 Corbin Ave, Northridge preferences completed
Target, 11051 Victory Blvd, North Hollywood preferences completed
Kohl's, 1201 S Fremont Ave, Alhambra preferences completed
Target, 5500 W Sunset Blvd, Los Angeles preferences completed
JCPenney, 20700 Avalon Blvd, Carson preferences completed
Target, 20700 Avalon Blvd Suite 700, Carson preferences completed
Target, 3535 S La Cienega Blvd, Los Angeles preferences completed
Target, 1601 Kingsdale Ave, Redondo Beach preferences completed
Target, 3471 W Century Blvd, Inglewood preferences completed
Kohl's, 25375 Crenshaw Blvd, Torrance preferences completed
Kohl's, 504 Huntington Dr, Monrovi

In [80]:
# save preferences
store_preferences.to_csv('data/stores/store_preferences.csv', index_label='store')
# save features dataframes
clear_df = pd.DataFrame([])
for key in businesses_dict:
    businesses_dict[key].loc[:, 'store'] = key
    clear_df = pd.concat([clear_df, businesses_dict[key]], axis=0)
    clear_df.to_csv('data/stores/business_communities.csv', index=False)
clear_df = pd.DataFrame([])
for key in highways_dict:
    highways_dict[key].loc[:, 'store'] = key
    clear_df = pd.concat([clear_df, highways_dict[key]], axis=0)
    clear_df.to_csv('data/stores/highways.csv')
clear_df = pd.DataFrame([])
for key in design_dict:
    design_dict[key]['store'] = key
    clear_df = pd.concat([clear_df, design_dict[key]], axis=0)
    clear_df.to_csv('data/stores/attractions.csv')
clear_df = pd.DataFrame([])
for key in road_dict:
    road_dict[key]['store'] = key
    clear_df = pd.concat([clear_df, road_dict[key]], axis=0)
    clear_df.to_csv('data/stores/roads.csv')
clear_df = pd.DataFrame([])
for key in walkways_dict:
    walkways_dict[key]['store'] = key
    clear_df = pd.concat([clear_df, walkways_dict[key]], axis=0)
    clear_df.to_csv('data/stores/walkways.csv', index=False)
clear_df = pd.DataFrame([])
for key in parking_dict:
    parking_dict[key]['store'] = key
    clear_df = pd.concat([clear_df, parking_dict[key]], axis=0)
    clear_df.to_csv('data/stores/parking_spaces.csv', index=False)